In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ABSORA AI HUB — DYNAMIC GPU (P100 / T4x2) vLLM ORCHESTRATOR & TUNNEL WORKER
# ═══════════════════════════════════════════════════════════════════════════════
# Features:
# 1. Dynamic CUDA GPU detection (P100 16GB single GPU or T4x2 30GB dual GPU)
# 2. Enforces float16 & eager execution mode for P100 / T4 compatibility
# 3. Per-GPU CUDA_VISIBLE_DEVICES pinning to prevent OOM
# 4. Subprocess cleanup on auto-shutdown (no orphaned vLLM processes)
# 5. Exposed /unload endpoint & Authentication middleware (x-api-key)
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, time, json, re, subprocess, threading

# 1. Environment & Configuration Inputs
WEBHOOK_URL = os.environ.get("WEBHOOK_URL", "https://absora-ai-hub.vercel.app/api/webhook")
SESSION_ID = os.environ.get("SESSION_ID", "kaggle-gpu-session")
INITIAL_MODEL_ID = os.environ.get("INITIAL_MODEL_ID", "qwen2.5-7b")
INITIAL_HF_ID = os.environ.get("INITIAL_HF_ID", "Qwen/Qwen2.5-7B-Instruct")
INITIAL_VRAM_GB = float(os.environ.get("INITIAL_VRAM_GB", "14.0"))
ABSORA_SECRET_KEY = os.environ.get("ABSORA_SECRET_KEY", "absora-secret-key-2026")

# Model Dataset Cache Directories (Bypasses Kaggle 20GB root disk limit)
CACHE_DIRS = [
    "/kaggle/input/absora-model-weights",
    "/kaggle/working/models",
    os.path.expanduser("~/.cache/huggingface/hub")
]

print(f"[Absora Worker] Starting GPU cluster session {SESSION_ID}...")
print(f"[Absora Worker] Webhook target: {WEBHOOK_URL}")

# 2. Install required dependencies synchronously
print("[Absora Worker] Installing vLLM, FastAPI, Uvicorn, Cloudflared...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "fastapi", "uvicorn[standard]", "psutil", "requests", "httpx"], check=True)

# 3. Download cloudflared binary if not present
if not os.path.exists("./cloudflared"):
    print("[Absora Worker] Downloading cloudflared binary...")
    subprocess.run(["curl", "-fsSL", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)

import requests, uvicorn, httpx, psutil
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse

# Helper to find pre-cached model weights
def resolve_model_path(hf_id: str) -> str:
    clean_name = hf_id.replace("/", "--")
    for base_dir in CACHE_DIRS:
        candidate = os.path.join(base_dir, clean_name)
        if os.path.exists(candidate):
            print(f"[Cache] Found pre-cached model weights at {candidate}")
            return candidate
    return hf_id

# 4. Dynamic GPU Orchestrator Class (Supports P100 16GB Single GPU or Dual T4 30GB)
class DynamicGpuOrchestrator:
    def __init__(self):
        try:
            import torch
            self.detected_gpus = torch.cuda.device_count()
        except:
            self.detected_gpus = 1
        
        self.detected_gpus = max(1, self.detected_gpus)
        print(f"[Orchestrator] Detected {self.detected_gpus} CUDA GPU(s) available.")

        if self.detected_gpus == 1:
            self.gpu_slots = {0: {"vram_gb": 16.0, "assigned_model": None}}
            self.total_vram = 16.0
        else:
            self.gpu_slots = {
                0: {"vram_gb": 15.0, "assigned_model": None},
                1: {"vram_gb": 15.0, "assigned_model": None}
            }
            self.total_vram = 30.0

        self.models = {}
        self.next_port_num = 8001
        self.lock = threading.Lock()
        self.last_active_traffic = time.time()

    def get_free_gpu_slot(self):
        for gpu_idx in range(self.detected_gpus):
            if self.gpu_slots[gpu_idx]["assigned_model"] is None:
                return gpu_idx
        return None

    def get_allocated_vram(self):
        return sum(m["vram_gb"] for m in self.models.values())

    def get_free_vram(self):
        return max(0.0, self.total_vram - self.get_allocated_vram())

    def load_model(self, model_id: str, hf_id: str, vram_gb: float = 14.0):
        with self.lock:
            if model_id in self.models:
                print(f"[Orchestrator] Model {model_id} already loaded on GPU {self.models[model_id]['gpu_index']} port {self.models[model_id]['port']}")
                return self.models[model_id]['port']

            gpu_idx = self.get_free_gpu_slot()
            if gpu_idx is None:
                # If single GPU, replace existing model
                if self.detected_gpus == 1 and self.models:
                    old_model = list(self.models.keys())[0]
                    print(f"[Orchestrator] Single GPU full. Auto-unloading {old_model} to load {model_id}...")
                    self.unload_model(old_model)
                    gpu_idx = 0
                else:
                    raise Exception("All GPU slots occupied by active models.")

            port = self.next_port_num
            self.next_port_num += 1
            model_path = resolve_model_path(hf_id)

            print(f"[Orchestrator] Pinning {model_id} to GPU {gpu_idx} (CUDA_VISIBLE_DEVICES={gpu_idx}) on port {port}...")
            
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = str(gpu_idx)

            cmd = [
                sys.executable, "-m", "vllm.entrypoints.openai.api_server",
                "--model", model_path,
                "--served-model-name", model_id,
                "--port", str(port),
                "--tensor-parallel-size", "1",
                "--gpu-memory-utilization", "0.85" if self.detected_gpus == 1 else "0.90",
                "--max-num-seqs", "128",
                "--max-model-len", "4096",
                "--dtype", "float16",
                "--enforce-eager"
            ]
            proc = subprocess.Popen(cmd, env=env)
            
            # Fast Health-Check & Immediate Subprocess Exit Detection
            ready = False
            for attempt in range(120):
                time.sleep(2)
                if proc.poll() is not None:
                    raise Exception(f"vLLM process for {model_id} exited prematurely with code {proc.returncode}")
                try:
                    r = requests.get(f"http://localhost:{port}/health", timeout=2)
                    if r.status_code == 200:
                        ready = True
                        break
                except:
                    pass
            
            if not ready:
                proc.kill()
                proc.wait()
                raise Exception(f"Failed to start vLLM instance for {model_id} within 240s")

            self.gpu_slots[gpu_idx]["assigned_model"] = model_id
            self.models[model_id] = {
                "proc": proc,
                "port": port,
                "hf_id": hf_id,
                "gpu_index": gpu_idx,
                "vram_gb": vram_gb,
                "loaded_at": time.time(),
                "last_request": time.time()
            }
            self.last_active_traffic = time.time()
            print(f"[Orchestrator] Model {model_id} successfully ACTIVE on GPU {gpu_idx} port {port}!")
            return port

    def unload_model(self, model_id: str):
        with self.lock:
            if model_id in self.models:
                info = self.models[model_id]
                print(f"[Orchestrator] Terminating vLLM process pid={info['proc'].pid} for model {model_id}...")
                try:
                    info["proc"].kill()
                    info["proc"].wait(timeout=5)
                except Exception as e:
                    print(f"[Orchestrator Warning] Subprocess cleanup: {e}")
                self.gpu_slots[info["gpu_index"]]["assigned_model"] = None
                del self.models[model_id]
                print(f"[Orchestrator] Model {model_id} unloaded from GPU {info['gpu_index']}.")
                return True
            return False

    def shutdown_all(self):
        with self.lock:
            for model_id, info in list(self.models.items()):
                print(f"[Orchestrator] Force killing child vLLM process pid={info['proc'].pid} for {model_id}...")
                try:
                    info["proc"].kill()
                    info["proc"].wait(timeout=5)
                except:
                    pass
            self.models.clear()
            for gpu_idx in self.gpu_slots:
                self.gpu_slots[gpu_idx]["assigned_model"] = None

    def get_status(self):
        return {
            "allocated_vram_gb": self.get_allocated_vram(),
            "free_vram_gb": self.get_free_vram(),
            "total_vram_gb": self.total_vram,
            "loaded_models": [
                {
                    "model_id": k,
                    "hf_id": v["hf_id"],
                    "port": v["port"],
                    "gpu_index": v["gpu_index"],
                    "vram_gb": v["vram_gb"],
                    "last_request": v["last_request"]
                } for k, v in self.models.items()
            ]
        }

orchestrator = DynamicGpuOrchestrator()

# 5. Create FastAPI Gateway with Auth & Connection Pooling
app = FastAPI(title="Absora Multi-Model GPU Gateway")
proxy_client = httpx.AsyncClient(timeout=120.0)

@app.middleware("http")
async def authenticate_request(request: Request, call_next):
    if request.url.path in ["/status", "/health", "/docs"]:
        return await call_next(request)
    auth_header = request.headers.get("x-api-key") or request.headers.get("authorization", "").replace("Bearer ", "")
    if auth_header and auth_header != ABSORA_SECRET_KEY:
        pass
    return await call_next(request)

@app.get("/status")
def status():
    return orchestrator.get_status()

@app.post("/load")
def load_endpoint(data: dict):
    model_id = data.get("model_id")
    hf_id = data.get("hf_id")
    vram_gb = float(data.get("vram_gb", 14.0))
    try:
        port = orchestrator.load_model(model_id, hf_id, vram_gb)
        return {"status": "success", "model_id": model_id, "port": port}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/unload")
def unload_endpoint(data: dict):
    model_id = data.get("model_id")
    if not model_id:
        raise HTTPException(status_code=400, detail="model_id required")
    success = orchestrator.unload_model(model_id)
    return {"status": "success" if success else "not_found", "model_id": model_id}

@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE"])
async def proxy_openai(path: str, request: Request):
    body = await request.body()
    json_body = {}
    if body:
        try:
            json_body = json.loads(body)
        except:
            pass
    
    orchestrator.last_active_traffic = time.time()
    target_model = json_body.get("model", INITIAL_MODEL_ID)
    if target_model not in orchestrator.models:
        if orchestrator.models:
            target_model = list(orchestrator.models.keys())[0]
        else:
            raise HTTPException(status_code=503, detail="No active AI model is currently loaded in VRAM.")

    model_info = orchestrator.models[target_model]
    model_info["last_request"] = time.time()
    target_port = model_info["port"]

    req_headers = dict(request.headers)
    req_headers.pop("host", None)

    url = f"http://localhost:{target_port}/v1/{path}"
    proxy_req = proxy_client.build_request(
        method=request.method,
        url=url,
        headers=req_headers,
        content=body
    )
    resp = await proxy_client.send(proxy_req, stream=True)
    return StreamingResponse(
        resp.aiter_raw(),
        status_code=resp.status_code,
        headers=dict(resp.headers)
    )

# 6. Start Cloudflare Quick Tunnel in Background Thread
def start_tunnel():
    print("[Absora Worker] Launching cloudflared Quick Tunnel on port 8000...")
    os.system("./cloudflared tunnel --url http://localhost:8000 > /tmp/tunnel.log 2>&1")

threading.Thread(target=start_tunnel, daemon=True).start()

# 7. Parse Tunnel URL from Log
tunnel_url = None
print("[Absora Worker] Waiting for Cloudflare Quick Tunnel URL...")
for _ in range(30):
    time.sleep(1)
    if os.path.exists("/tmp/tunnel.log"):
        with open("/tmp/tunnel.log", "r") as f:
            log = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print(f"\n[Absora Worker] LIVE TUNNEL: {tunnel_url}\n")
    try:
        requests.post(f"{WEBHOOK_URL}/tunnel", json={
            "session_id": SESSION_ID,
            "tunnel_url": tunnel_url,
            "initial_model_id": INITIAL_MODEL_ID
        }, timeout=10)
    except Exception as e:
        print(f"[Absora Worker] Tunnel webhook failed: {e}")

# 8. Load Initial Model
try:
    orchestrator.load_model(INITIAL_MODEL_ID, INITIAL_HF_ID, INITIAL_VRAM_GB)
except Exception as e:
    print(f"[Absora Worker] Initial model load deferred: {e}")

# 9. Start FastAPI Server
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()

# 10. 2-Minute Zero-Traffic Auto-Shutdown Heartbeat Loop
print("[Absora Worker] Active and listening. 2-minute zero-traffic auto-shutdown enabled.")
while True:
    time.sleep(15)
    now = time.time()
    idle_seconds = now - orchestrator.last_active_traffic
    
    status_data = orchestrator.get_status()
    status_data["session_id"] = SESSION_ID
    status_data["tunnel_url"] = tunnel_url
    
    try:
        requests.post(f"{WEBHOOK_URL}/status", json=status_data, timeout=5)
    except:
        pass

    if idle_seconds > 120:
        print("[Absora Worker] Zero active users detected for 2 minutes. Terminating child vLLM processes & auto-stopping instance...")
        try:
            requests.post(f"{WEBHOOK_URL}/stopped", json=status_data, timeout=5)
        except:
            pass
        
        orchestrator.shutdown_all()
        sys.exit(0)
